# 00 — Lines as Paths

In geometry, a line extends infinitely in both directions. In spatial data, we never use infinite lines — we use **line segments**: paths that start somewhere and end somewhere.

This notebook establishes how a path is represented — as an ordered list of coordinates — and how that structure maps to a GeoJSON `LineString`. Every route, trajectory, and boundary in this module is a variation of this pattern.

```
coordinates  →  LineString geometry  →  Feature  →  map layer
```

## Setup

In [1]:
import json
from pathlib import Path
from ipyleaflet import Map, GeoJSON, basemaps

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

## A Path is a Sequence of Points

The simplest path has two endpoints: a start and a finish.

```python
path = [start, end]
```

Each point is a `[longitude, latitude]` pair — GeoJSON always uses `[lon, lat]` order. Three or more points describe a multi-leg route, each consecutive pair forming one **segment** of the full path.

```
[A] ———— [B] ———— [C] ———— [D]
 ↑                          ↑
start                      end
      ↑        ↑        ↑    
           3 segments        
```

The order matters — reversing the list reverses the direction of travel.

In [2]:
# The most basic path: two endpoints
launch = [-77.009,  38.889]   # [lon, lat] — Washington D.C.
target = [ 51.388,  35.695]   # [lon, lat] — Tehran, Iran

path = [launch, target]

print("Start:   ", path[0])
print("End:     ", path[1])
print("Segments:", len(path) - 1)

Start:    [-77.009, 38.889]
End:      [51.388, 35.695]
Segments: 1


## LineString in GeoJSON

GeoJSON defines a geometry type called `LineString` for exactly this: an ordered sequence of two or more positions.

```json
{
  "type": "LineString",
  "coordinates": [
    [lon1, lat1],
    [lon2, lat2],
    ...
  ]
}
```

Wrapped in a `Feature` with `properties`, it becomes a full GeoJSON object that any mapping library can render directly.

In [3]:
# Build a complete LineString Feature
route = {
    "type": "Feature",
    "properties": {
        "name": "Alpha",
        "origin": "Washington D.C.",
        "destination": "Tehran"
    },
    "geometry": {
        "type": "LineString",
        "coordinates": [
            [-77.009,  38.889],   # Washington D.C.
            [ 51.388,  35.695],   # Tehran
        ]
    }
}

print("Feature type:  ", route["type"])
print("Geometry type: ", route["geometry"]["type"])
print("Point count:   ", len(route["geometry"]["coordinates"]))
print("Name:          ", route["properties"]["name"])

Feature type:   Feature
Geometry type:  LineString
Point count:    2
Name:           Alpha


## Line vs. Line Segment

| Concept | Description | Used in mapping? |
|---|---|---|
| **Line** | Extends infinitely in both directions | No |
| **Ray** | Starts at a point, extends infinitely in one direction | Rarely |
| **Line segment** | Bounded by two endpoints | Yes — always |

Every `LineString` in GeoJSON is a chain of **segments**. When we later ask "does this path cross that polygon?", we are asking whether any segment of the path crosses any edge of the polygon.

The infinite-line formulas are useful mathematically — the intersection test works by extending both lines to find where they *would* meet — but we then check whether that meeting point falls within the bounded extent of both segments. If it does not, the segments do not intersect even though their containing lines do.

## Multi-Point Paths

A trajectory rarely travels in a perfectly straight line between distant points. A multi-point path adds **waypoints** — intermediate coordinates that bend or redirect the route.

Each consecutive pair of points is one segment. A path with `n` points has `n − 1` segments.

In [4]:
# A multi-leg path: 4 waypoints, 3 segments
waypoints = [
    [-77.009,  38.889],   # Washington D.C.
    [ 10.000,  50.000],   # Central Europe (waypoint)
    [ 37.617,  55.755],   # Moscow
    [ 51.388,  35.695],   # Tehran
]

n_points   = len(waypoints)
n_segments = n_points - 1

print(f"{n_points} points → {n_segments} segments")
for i in range(n_segments):
    a = waypoints[i]
    b = waypoints[i + 1]
    print(f"  Segment {i + 1}: {a}  →  {b}")

4 points → 3 segments
  Segment 1: [-77.009, 38.889]  →  [10.0, 50.0]
  Segment 2: [10.0, 50.0]  →  [37.617, 55.755]
  Segment 3: [37.617, 55.755]  →  [51.388, 35.695]


## Visualizing a Path on a Map

A `LineString` Feature loads directly into ipyleaflet's `GeoJSON` layer. Wrapping it in a `FeatureCollection` keeps the pattern consistent — you can always add more features later without changing the rendering code.

In [5]:
path_fc = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {"name": "Alpha"},
            "geometry": {
                "type": "LineString",
                "coordinates": waypoints
            }
        }
    ]
}

m = Map(center=(45, 0), zoom=2, basemap=basemaps.CartoDB.Positron)
m.add(GeoJSON(
    data=path_fc,
    style={"color": "#e74c3c", "weight": 2, "opacity": 0.85}
))
m

Map(center=[45, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

## Saving the Module's Path Dataset

The notebooks that follow all work from the same set of paths. We define them here and write them to `data/paths.geojson` so every subsequent notebook loads one consistent file.

Each path has a `name`, `origin`, and `destination` in its properties — enough context to identify which routes crossed which regions when we run intersection tests later.

In [6]:
module_paths = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {"name": "Alpha", "origin": "Washington D.C.", "destination": "Tehran"},
            "geometry": {
                "type": "LineString",
                "coordinates": [
                    [-77.009,  38.889],   # Washington D.C., USA
                    [ 51.388,  35.695],   # Tehran, Iran
                ]
            }
        },
        {
            "type": "Feature",
            "properties": {"name": "Bravo", "origin": "Pyongyang", "destination": "Honolulu"},
            "geometry": {
                "type": "LineString",
                "coordinates": [
                    [ 125.738,  39.020],   # Pyongyang, North Korea
                    [-157.855,  21.305],   # Honolulu, Hawaii, USA
                ]
            }
        },
        {
            "type": "Feature",
            "properties": {"name": "Charlie", "origin": "Caracas", "destination": "Madrid"},
            "geometry": {
                "type": "LineString",
                "coordinates": [
                    [-66.879,  10.488],   # Caracas, Venezuela
                    [ -3.703,  40.417],   # Madrid, Spain
                ]
            }
        },
        {
            "type": "Feature",
            "properties": {"name": "Delta", "origin": "Moscow", "destination": "Riyadh"},
            "geometry": {
                "type": "LineString",
                "coordinates": [
                    [ 37.617,  55.755],   # Moscow, Russia
                    [ 46.675,  24.688],   # Riyadh, Saudi Arabia
                ]
            }
        }
    ]
}

out_path = DATA_DIR / "paths.geojson"
with open(out_path, "w") as f:
    json.dump(module_paths, f, indent=2)

print(f"Saved {len(module_paths['features'])} paths → {out_path}")
for feat in module_paths["features"]:
    p = feat["properties"]
    n = len(feat["geometry"]["coordinates"])
    print(f"  {p['name']:8}  {p['origin']} → {p['destination']}  ({n} points, {n-1} segment{'s' if n-1 > 1 else ''})")

Saved 4 paths → data/paths.geojson
  Alpha     Washington D.C. → Tehran  (2 points, 1 segment)
  Bravo     Pyongyang → Honolulu  (2 points, 1 segment)
  Charlie   Caracas → Madrid  (2 points, 1 segment)
  Delta     Moscow → Riyadh  (2 points, 1 segment)


## Exercise A

A direct flight from **Los Angeles** (`[-118.243, 34.052]`) to **London** (`[-0.127, 51.507]`) is routed through a waypoint over **Reykjavik, Iceland** (`[-22.000, 64.133]`).

1. Build a `LineString` Feature with all three points and a `"name"` property of `"LA to London"`.
2. Wrap it in a `FeatureCollection` and display it on a world map centered at `(50, -50)`, zoom 3.
3. Print the number of segments.

In [ ]:
from ipyleaflet import Map, GeoJSON, basemaps

route = {
    'type': 'Feature',
    'properties': {'name': 'LA to London'},
    'geometry': {
        'type': 'LineString',
        'coordinates': [
            [-118.243, 34.052],
            [-22.000, 64.133],
            [-0.127, 51.507],
        ],
    },
}
route_fc = {'type': 'FeatureCollection', 'features': [route]}

m_ex = Map(center=(50, -50), zoom=3, basemap=basemaps.CartoDB.Positron)
m_ex.add(GeoJSON(
    data=route_fc,
    style={'color': '#e74c3c', 'weight': 2.5, 'opacity': 0.9},
))
print('Segments:', len(route['geometry']['coordinates']) - 1)
m_ex

## Exercise B

You are given a list of `(lat, lon)` tuples — note the **lat/lon order**, which is the opposite of GeoJSON's `[lon, lat]`.

```python
waypoints_latlon = [
    (48.857,   2.352),   # Paris
    (41.890,  12.492),   # Rome
    (37.983,  23.727),   # Athens
    (41.015,  28.979),   # Istanbul
    (30.044,  31.236),   # Cairo
]
```

1. Convert the list to GeoJSON `[lon, lat]` coordinate order.
2. Build a `LineString` Feature with `"name": "Mediterranean Arc"`.
3. Save it as `data/med_arc.geojson`.
4. Load the file back and print the coordinate count to confirm the round-trip.

In [ ]:
import json
from pathlib import Path

waypoints_latlon = [
    (48.857,   2.352),   # Paris
    (41.890,  12.492),   # Rome
    (37.983,  23.727),   # Athens
    (41.015,  28.979),   # Istanbul
    (30.044,  31.236),   # Cairo
]

coords = [[lon, lat] for lat, lon in waypoints_latlon]
med_arc = {
    'type': 'Feature',
    'properties': {'name': 'Mediterranean Arc'},
    'geometry': {'type': 'LineString', 'coordinates': coords},
}

out_path = DATA_DIR / 'med_arc.geojson'
with open(out_path, 'w') as f:
    json.dump({'type': 'FeatureCollection', 'features': [med_arc]}, f, indent=2)

with open(out_path) as f:
    reloaded = json.load(f)

count = len(reloaded['features'][0]['geometry']['coordinates'])
print(f'Saved to {out_path}')
print(f'Coordinate count after round-trip: {count}')

---

## Check Your Understanding

**1.** What is the difference between a line and a line segment, and why does the distinction matter for GeoJSON?

**2.** Why do we use segments instead of infinite lines in spatial data?

```python
# No code needed — answer in your own words
```

## Next

In [01 — Line Segment Intersection](./01-Line_Segment_Intersection.ipynb), we ask when two segments actually cross — building the orientation-based test that drives every intersection check in this module.